# CS5489 Phase 5：原图裁块与掩码感知采样

## 依据与前置检查

[NCL-IML，ICCV 2023](https://arxiv.org/abs/2309.14900) 在局部特征上区分编辑和真实区域。[Liu 等，IEEE TIFS 2024](https://www.ying-cao.com/projects/image_manipulation/files/image_manipulation_tifs2024.pdf) 研究边界引导的特征采样、注意力和对比学习。[IMDL-BenCo，NeurIPS 2024](https://proceedings.neurips.cc/paper_files/paper/2024/hash/f280a398c243b5fdaa09f57ece880fc9-Abstract-Datasets_and_Benchmarks_Track.html) 强调统一训练与评价协议。本轮只借鉴局部区域、边界与困难背景的重要性，把比较限定为输入裁块的选择规则；模型与损失中没有加入这些论文的对比学习或边界模块。

V1 用 118,273 参数的 SmallUNet，Encoder 通道 16、32，瓶颈 64，Decoder 32、16 并拼接跳连。PyTorch 默认随机初始化，种子 5489，Adam 0.001，批大小 8，无调度器，损失为无权 BCEWithLogits 加逐图 soft Dice。Phase 4 复核了 V1 最佳权重在训练和验证集的 256-grid IoU，分别为 13.19% 和 12.99%。8 张固定训练样本的短实验 IoU 到 71.11%，说明基本训练流程能学到标签。[V1 源码](../baseline/project.py) · [Phase 4 诊断](../phase4/preflight.json)

V1 训练损失 1.2665→1.2000，验证损失第 6 轮最低 1.2042，最后为 1.2113，验证 IoU 第 9 轮 12.99%、第 10 轮 12.18%。这些曲线支持在当前表示下训练和验证都偏弱，不支持简单归因于“再训几轮就能提高”。Phase 2 的加权主要交换精确率和召回率，Phase 3 的提高尺寸没改善全局 IoU，Phase 4 的固定残差降低了 IoU 和共同网格 PR-AUC。Phase 5 沿用 V1 的 RGB 和损失。[V1 曲线](../baseline/training_history.csv) · [Phase 2 报告](../phase2/Phase2_诊断与实验报告.md) · [Phase 3 报告](../phase3/Phase3_分辨率实验报告.md) · [Phase 4 报告](../phase4/Phase4_SRM实验报告.md)

| 训练集原图统计 | 最小值 | P05 | 中位数 | P95 | 最大值 |
|---|---:|---:|---:|---:|---:|
| 短边（像素） | 518 | 680 | 1278 | 3168 | 4079 |
| 长边（像素） | 563 | 1024 | 1800 | 3185 | 4079 |
| 编辑区域等效方形边长（面积的平方根，像素） | 20.2 | 84.5 | 333.8 | 1365.0 | 3599.9 |
| 真实前景比例 | 0.015% | 0.36% | 4.73% | 48.27% | 100% |

训练集有 764 张原图的编辑区占比小于 1%，全部训练图都可直接裁 256×256 原始像素。等效边长只描述面积，细长编辑区可能跨越多个窗口。裁块保留原始局部细节，但单窗看不到远处背景、重复物体或大区域整体一致性。滑窗融合只能处理重叠处的局部预测，不能向模型补充窗外上下文。

尺寸不足时对称补边，RGB 用反射填充，单边长度为 1 时用端点复制，掩码补零。RGB 和 Mask 使用同一坐标，选点和裁剪都在同一图上。预检覆盖了五组尺寸的坐标还原与常数融合，并用 V1 的已有权重检查三张真实图的原尺寸和 RLE 往返。[数据尺寸](data_preflight.json) · [合成滑窗检查](sliding_preflight.json) · [真实图滑窗检查](real_inference_preflight.json)

## 固定实验设计和实际采样

- 每组 10 轮，每轮 4,500 张训练图各裁一块，批大小 8，共 563 步/轮、5,630 步/组。两组重置相同种子，复用 V1 的模型、损失和优化器，初始化哈希应相同。A 的左上角在合法坐标中均匀随机。B 中指定为均匀抽样的样本与 A 在同源同轮的坐标一致。
- B 每轮固定指派 2,250 块均匀、1,350 块编辑区、900 块困难背景。编辑区先从原图掩码的粗网格边界选择候选中心并做坐标抖动，最多 12 次找前景占比 1%–95% 的裁块。否则用已试到的任意非空块，再失败则用 A 的均匀坐标。背景分支在 12 个均匀候选内挑选掩码全零、绿色通道邻域变化最大的块，无合适块时回退到均匀。验证掩码不进入训练裁块生成。
- A 有 14,703/45,000 块含前景。B 有 21,306/45,000 块含前景，13,500 块指定编辑裁块中有 24 块走任意非空回退。B 指定的 9,000 块困难背景中，8,597 块成功选到全背景纹理块，403 块回退均匀裁块，其中 390 块含前景。成功背景块的平均纹理强度为 12.64，A 的均匀块为 8.06。实际前景比例和回退次数记录在日志中。
- 两组每轮用同一套确定性的 2,500 验证裁块（500 张图各五个固定坐标）计算 pooled IoU@0.5，选最大值的最早轮次。最终只对保存的最佳权重运行全部 500 张验证图的原图滑窗。
- 滑窗为 256×256、固定步长 192、推理 batch 32，末窗贴边。把每窗 sigmoid 概率用二维可分离 Hann 权重融合，每轴权重下限 0.1，确保边缘仍有正权重。还原原尺寸后统一用阈值 0.5，导出课程模板的列优先 RLE。

[采样摘要](sampling_summary.csv) · [逐块坐标、前景面积及回退记录](sampling_by_crop.csv) · [两组同源裁块示例](sampling_examples.png) · [采样代码](prepare.py) · [滑窗实现](sliding.py)

## 课程交付

课件确认 Canvas 截止时间为 **2026-12-13 23:59:59**。项目报告 Notebook 需包含可复现的训练代码、运行输出、观察和分析，另交 test.ipynb、model.pth 和 Python 环境说明。再次查阅了课程的 Kaggle 邀请页（课程渠道），仍未能公开取得 Evaluation 或 Rules。官方评分公式、外部模型与预训练许可仍待确认，TA 模板两列表头与数据包的五列表头差异也待竞赛页核对。本轮仅用课程数据、从头训练。[课件第 4–6 页](../Lecture7_1.pdf#page=4) · [已有规则核对](../phase2/Phase2_诊断与实验报告.md)

## 原图结果

两个模型以阈值 0.5 评价同一 500 张原图。micro 汇总全部前景像素计数，逐图平均则每张图等权。PR-AUC 使用这 500 张原图全部像素的预测概率，分到 4,096 个等宽区间并对汇总 PR 曲线作梯形积分，因此是近似值；AP 也另存。Phase 4 的 PR-AUC 用共同 256 网格，口径不同。

| 指标 | A 均匀裁块 | B 掩码感知 | B−A |
|---|---:|---:|---:|
| 原图 micro-IoU | 9.79% | 10.51% | +0.71 个百分点 |
| 原图逐图平均 IoU | 9.99% | 11.75% | +1.76 个百分点 |
| 原图 Dice | 17.84% | 19.01% | +1.17 个百分点 |
| 原图 Precision | 40.09% | 40.52% | +0.42 个百分点 |
| 原图 Recall | 11.47% | 12.42% | +0.95 个百分点 |
| 原图近似 PR-AUC | 18.87% | 20.02% | +1.15 个百分点 |
| 预测前景占比 | 3.74% | 4.01% | +0.27 个百分点 |
| FP | 38,654,913 | 41,123,995 | +2,469,082 |
| FN | 199,641,669 | 197,499,745 | −2,141,924 |
| 最佳轮次与固定裁块 IoU | 8 / 10.13% | 8 / 11.45% | +1.32 个百分点 |
| 10 轮训练与裁块验证 | 97.60 s | 92.70 s | −4.90 s |
| 训练峰值 PyTorch 分配显存 | 593.41 MiB | 593.41 MiB | 0 |
| 原图滑窗总推理 | 32.51 s | 19.13 s | A 包含首次启动 |
| 去掉首张图后的滑窗总推理 | 18.45 s | 19.11 s | +0.67 s |
| 单图滑窗中位耗时 | 26.03 ms | 25.81 ms | −0.22 ms |

两组各推理 48,093 个窗口，单图中位 63 个。滑窗计时包含单张图张量准备、模型、融合和拷回 CPU，不含 WebP 解码与指标统计。A 的首张图耗时 14.06 s，B 为 0.02 s，不能把原始总时间差归因于采样规则。稳态约 26 ms/图是本机 RTX 5090 的实测，交付环境需要另做计时。Micro-IoU 低于全前景的 13.07%，也低于整图缩放历史结果中的 V1 13.36%、加权 BCE 14.05%、384 版 13.20%、SRM 11.35%。历史结果的训练输入、测试视野与本轮不同，不能从横向名次单独推断裁块导致下降。[A 逐图](A/validation_by_image.csv) · [B 逐图](B/validation_by_image.csv) · [主指标记录](validation_summary.json) · [PR 曲线](pr_curves.png)

| 原图真实前景面积 | 图数 | A micro-IoU | B micro-IoU |
|---|---:|---:|---:|
| <1% | 73 | 1.04% | 1.41% |
| 1%–<5% | 171 | 10.37% | 12.17% |
| 5%–<15% | 139 | 7.92% | 9.69% |
| 15%–<30% | 52 | 20.04% | 19.94% |
| 30%–100% | 65 | 7.91% | 8.26% |

### 已验证的变化与统计区间

按图片做 3,000 次配对 Bootstrap，在同一次抽样中 A、B 共用图片下标，并重新汇总 TP、FP、FN。全验证集 B−A 的 95% 区间如下，单位为百分点。

| 指标 | 差值 | 95% 配对区间 |
|---|---:|---:|
| 全集 micro-IoU | +0.71 | +0.12 至 +1.31 |
| 全集逐图平均 IoU | +1.76 | +1.06 至 +2.47 |
| 全集 Dice | +1.17 | +0.20 至 +2.13 |
| 全集 Recall | +0.95 | +0.29 至 +1.65 |
| 全集背景 FPR | +0.16 | −0.28 至 +0.61 |
| <1% 组 micro-IoU | +0.37 | −0.70 至 +2.04 |
| <1% 组 Recall | +3.97 | −0.40 至 +11.31 |
| <1% 组背景 FPR | +0.72 | +0.16 至 +1.35 |

B 在本次固定权重对照的总体 IoU、Dice 和召回上有改善依据。小目标 IoU 和召回的区间跨零，因此不能判定 73 张小目标获得可靠增益。B 的总体误报像素增加，全集 FPR 区间跨零，小区域图里的背景 FPR 则有上升迹象。区间只描述同一验证集上固定模型的图片抽样不确定性，不能覆盖另一个训练种子的变化。[面积分组](area_bins.csv) · [完整配对区间](bootstrap.json)

按固定绿色邻域差代理测量复杂背景（原图每隔两像素采样，水平或垂直差至少 20），B 对高纹理背景的 FPR 从 2.00% 降到 1.64%，低纹理背景 FPR 从 2.66% 升到 2.91%。高纹理组没有观察到误报增加；代理只覆盖一种纹理定义，也不能把差异单独归因于 B 的背景分支，因为 B 同时改变了正块和负块的选择。图 001882 的小编辑 IoU 从 0.129 到 0.260，但 004657 两组都没有有效交集。30% 以上大区域两组的 Recall 只有 8.10%、8.48%。局部视野不足是合理解释，当前设计无法把它与输入尺度或其他学习因素分离。[同图预测](validation_examples.png)

### 阈值诊断与训练后段

九档验证扫描里 A 在 0.1 最佳，IoU 13.0691%，基本等于全前景。B 在 0.2 最佳，IoU 13.0703%，同样几乎没超过参照。主比较固定 0.5。扫描是在选模所用同一批验证图上完成，最佳值可能偏乐观，未用公测集调参。[九档阈值](threshold_sweep.csv)

A 的固定验证裁块 IoU 从第 8 轮的 10.13% 降至第 10 轮的 5.94%，B 从 11.45% 降至 10.65%。训练 loss 后段仍略降，但验证 IoU 没持续改善，现有曲线不能说明单纯延长训练即可补足差距。[A 训练日志](A/training.log) · [B 训练日志](B/training.log) · [训练曲线](training_curves.png)

## 判断与下一步

本轮能支持的结论是掩码感知采样在这套裁块流程中优于均匀采样；两个模型均未达到全前景或既有整图模型的原图 IoU，尚未证明纯裁块路线有效。交付优先冻结可运行的整图候选，保留 V1 和 Weighted BCE，等官方评分口径确认后在固定验证集按该指标选一份模型。保留时间给跨机器的 TA Notebook 验证、报告和环境说明。

课程课件把目标定义为整图上的逐像素预测。结合本轮大区域的低召回，后续若有时间，优先审阅一个**保留整图上下文**的对照，例如在共同的 RGB 整图预测上固定使用既有局部模型做细化，比较同一整图模型有无局部补充。当前证据不足以为纯裁块增加轮数或优先上边界头。本阶段没有启动这些实验。[课件第 3 页](../Lecture7_1.pdf#page=3)

## 交接文件

- [A 权重](A/model.pth) · [A 配置](A/config.json) · [B 权重](B/model.pth) · [B 配置](B/config.json)
- [实验报告 Notebook](project_report.ipynb) · [TA 推理 Notebook](test.ipynb) · [Python 环境说明](python_environment.txt)
- [采样](prepare.py) · [真实图预检](inference_preflight.py) · [训练](train.py) · [原图评估](evaluate.py)

使用 cs5489，保持 Phase 5 目录和 baseline 在同一个 Course_Project_export 下。复现顺序为 `prepare.py`、`inference_preflight.py`、`train.py`、`evaluate.py`。已有训练日志的目录会拒绝覆盖；Notebook 中另有全新输出目录的复现入口。日常汇报可直接读已有的 CSV、JSON 和模型权重，不必复制约 24 GB 的裁块缓存；有正式 ZIP 时可重建。TA 推理可用 A 或 B 权重，统一输出课程模板要求的原图尺寸掩码及 RLE。


## Reproduction

Open this notebook in the Phase 5 directory with cs5489. Leave `REPRODUCE=False` to inspect saved outputs. The opt-in run writes fresh artifacts in a sibling directory and requires the original course archive and baseline files.

In [ ]:
from pathlib import Path
from shutil import copy2
import subprocess
import sys

stage = Path.cwd()
REPRODUCE = False
if REPRODUCE:
    fresh = stage.parent / 'phase5_reproduction'
    fresh.mkdir()
    scripts = ['prepare.py', 'inference_preflight.py', 'sliding.py', 'train.py', 'evaluate.py']
    for script in scripts:
        copy2(stage / script, fresh / script)
    for script in ['inference_preflight.py', 'prepare.py', 'train.py', 'evaluate.py']:
        subprocess.run([sys.executable, '-u', script], cwd=fresh, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import display

records = json.loads((stage / 'validation_summary.json').read_text(encoding='utf-8'))
fields = ['iou', 'macro_iou', 'precision', 'recall', 'dice', 'pr_auc_binned', 'predicted_fraction', 'FP', 'FN']
frame = pd.DataFrame.from_dict(records, orient='index')[fields]
for key in fields[:-2]:
    frame[key] = frame[key].map(lambda v: f'{v:.2%}')
display(frame)


In [ ]:
from IPython.display import Image, display
for name in ('training_curves.png', 'pr_curves.png'):
    display(Image(filename=str(stage / name)))
